# DiffusionGemma-Jev (djev) 

This notebook demonstrates using the bleeding-edge vLLM native features for Diffusion Forcing reads. You can run this natively on Colab's GPU or connect to your deployed Cloud Run endpoint.

In [1]:
!pip install -U vllm --pre --extra-index-url https://wheels.vllm.ai/nightly

In [2]:
import json
import math
import time
import urllib.request

# If you deployed to Cloud Run, paste your URL here (e.g., https://djev-dgemma-...run.app)
# Otherwise, we will use local vLLM pipeline (see below cell)
CLOUD_RUN_URL = ""

if CLOUD_RUN_URL:
    SCAFFOLD = [100, 45518, 107, 101]  # <thought>\n</thought>
    template_str = "department: a\nurgency: 1\nrefund_requested: yes"

    # 1. Tokenize template via POST /tokenize
    tok_req = urllib.request.Request(
        f"{CLOUD_RUN_URL}/tokenize",
        data=json.dumps({"prompt": template_str, "add_special_tokens": False}).encode(),
        headers={"Content-Type": "application/json"},
    )
    base_ids = json.loads(urllib.request.urlopen(tok_req).read().decode())["tokens"]
    full_template = SCAFFOLD + base_ids

    unpinned = [i - 1 for i, t in enumerate(full_template) if i >= 4 and t == 107] + [len(full_template) - 1]
    pinned = [i for i in range(len(full_template)) if i not in unpinned]
    seed_canvas = [full_template[i] if i in pinned else (256000 + i * 131) for i in range(len(full_template))]

    # 2. Run 1-step read-only diffusion forward pass via POST /v1/chat/completions
    chat_payload = {
        "model": "djev-dgemma",
        "messages": [
            {
                "role": "system",
                "content": (
                    "Answer each question about the ticket state with its single label.\n"
                    "department: a = billing, b = technical, c = sales\n"
                    "urgency: 1 = low, 2 = minor, 3 = locked production access, 4 = complete outage\n"
                    "refund_requested: yes or no"
                ),
            },
            {
                "role": "user",
                "content": json.dumps({
                    "ticket_id": "TCK-9042",
                    "text": "I was double-charged $149.00 on invoice INV-2026-8841. Please refund the duplicate charge.",
                }),
            },
        ],
        "max_tokens": len(full_template) + 1,
        "logprobs": True,
        "top_logprobs": 16,
        "extra_body": {
            "vllm_xargs": {
                "diffusion_seed_canvas": seed_canvas,
                "diffusion_pinned": pinned,
                "diffusion_max_steps": 1,
                "diffusion_read_only": True,
            }
        },
    }

    t0 = time.time()
    req = urllib.request.Request(
        f"{CLOUD_RUN_URL}/v1/chat/completions",
        data=json.dumps(chat_payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    resp = json.loads(urllib.request.urlopen(req).read().decode())
    rtt_ms = round((time.time() - t0) * 1000, 1)
    content_lp = resp["choices"][0]["logprobs"]["content"]
    print(f"Completed in {rtt_ms} ms via Cloud Run.")
else:
    print("No Cloud Run URL provided. Run the local vLLM pipeline below.")


In [3]:
from vllm import LLM, SamplingParams

if not CLOUD_RUN_URL:
    # Native Colab Pipeline
    llm = LLM(model="nvidia/diffusiongemma-26B-A4B-it-NVFP4", trust_remote_code=True, gpu_memory_utilization=0.9)
    tokenizer = llm.get_tokenizer()

    SCAFFOLD = [100, 45518, 107, 101] # thought scaffold
    template_str = "department: a\nurgency: 1\nrefund_requested: yes"
    base_ids = tokenizer.encode(template_str, add_special_tokens=False)
    full_template = SCAFFOLD + base_ids

    unpinned = [i - 1 for i, t in enumerate(full_template) if i >= 4 and t == 107] + [len(full_template) - 1]
    pinned = [i for i in range(len(full_template)) if i not in unpinned]
    seed_canvas = [full_template[i] if i in pinned else (256000 + i * 131) for i in range(len(full_template))]

    sampling_params = SamplingParams(
        max_tokens=len(full_template) + 1,
        logprobs=16,
        extra_args={
            "diffusion_seed_canvas": seed_canvas,
            "diffusion_pinned": pinned,
            "diffusion_max_steps": 1,
            "diffusion_read_only": True,
        }
    )
    
    messages = [
      {"role": "system", "content": "Answer each question about the ticket state with its single label..."},
      {"role": "user", "content": "I was double-charged $149.00 on invoice INV-2026-8841. Please refund the duplicate charge."}
    ]
    
    outputs = llm.chat(messages, sampling_params=sampling_params)
    print("Local Generation Completed:")
    for out in outputs:
        print(out.outputs[0].logprobs)
